## Dreambooth LoRA SDXL

In [ ]:
import gc
import json

import torch
from diffusers import (
    BitsAndBytesConfig,
    StableDiffusionXLPipeline,
    UNet2DConditionModel,
)
from IPython.display import display

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
BNB_QUANTIZATION_CONFIG_PATH = "configs/bnb/train_dreambooth_lora_sdxl.json"

In [ ]:
with open(BNB_QUANTIZATION_CONFIG_PATH, "r") as f:
    config_kwargs = json.load(f)
if config_kwargs.get("load_in_4bit"):
    config_kwargs["bnb_4bit_compute_dtype"] = torch.bfloat16
quantization_config = BitsAndBytesConfig(**config_kwargs)
for k, v in config_kwargs.items():
    print(f"{k}: {v}")

### Dog Toy Example

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"

In [ ]:
text_pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    unet=None,
    vae=None,
    scheduler=None,
    torch_dtype=torch.bfloat16,
    variant="fp16",
)

In [ ]:
prompt = "a photo of sks dog"
negative_prompt = ""

In [ ]:
with torch.inference_mode():
    (
        prompt_embeds,
        negative_prompt_embeds,
        pooled_prompt_embeds,
        negative_pooled_prompt_embeds,
    ) = text_pipe.encode_prompt(
        prompt=prompt,
        prompt_2=prompt,
        negative_prompt=negative_prompt,
        negative_prompt_2=negative_prompt,
    )

In [ ]:
del text_pipe

gc.collect()
torch.cuda.empty_cache()

In [ ]:
LORA_CHECKPOINT = "trained-sdxl-lora/checkpoint-1000"

In [ ]:
unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID,
    subfolder="unet",
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    variant="fp16",
)
unet.to("cuda")

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    unet=unet,
    tokenizer=None,
    tokenizer_2=None,
    text_encoder=None,
    text_encoder_2=None,
    torch_dtype=torch.bfloat16,
    variant="fp16",
)
pipe.load_lora_weights(LORA_CHECKPOINT)

In [ ]:
with torch.inference_mode():
    latents = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        pooled_prompt_embeds=pooled_prompt_embeds,
        negative_pooled_prompt_embeds=negative_pooled_prompt_embeds,
        height=1024,
        width=1024,
        num_inference_steps=50,
        guidance_scale=5.0,
        generator=torch.Generator("cuda").manual_seed(42),
        output_type="latent",
    ).images

In [ ]:
del unet

gc.collect()
torch.cuda.empty_cache()

In [ ]:
latents = latents / pipe.vae.config.scaling_factor
latents = latents.to(device="cpu", dtype=torch.bfloat16)
pipe.vae.to(device="cpu", dtype=torch.bfloat16)
with torch.inference_mode():
    image = pipe.vae.decode(latents, return_dict=False)[0]
image = pipe.image_processor.postprocess(image, output_type="pil")[0]

In [ ]:
display(image)